# E1.6 · Operating vs outcome guardrails

**Function E — AI Governance for Agentic Systems → Building the Governance Framework — Risk and Control**  ·  *Security of AI*

Builds on **[E1.5 · Evaluation output as audit evidence](https://spbreed.github.io/cyber-commons/lessons/E1.5.html)**.

| | |
|---|---|
| Tools used | NeMo Guardrails, LLM Guard, Llama Guard 4, Claude Haiku 4.5 |

## What this lesson is

**What it covers.** Classify your own guardrails into the two buckets.

**Why a security engineer needs it.** Frameworks specify how the system works; regulators care what it produced. The control it builds is: constrain both, and know which evidence answers which question.

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

Two different kinds of control get confused constantly. One bounds how the system runs — budgets, scopes, approvals. The other bounds what it produces. They are tested differently and they fail differently.

> **At CyberTravels.** Two different controls get confused: what bounds how CyberTravels runs (budgets, scopes, approvals) and what bounds what it produces (the hotel recommendation). They fail differently and are tested differently.

## 2 · The framework

```
   operating guardrails            outcome guardrails
   +----------------------+        +-----------------------+
   | HOW it runs          |        | WHAT it produces      |
   | budgets, scopes,     |        | content, decisions,   |
   | approvals, sandbox   |        | actions taken         |
   +----------------------+        +-----------------------+
   tested by attempting    tested by sampling outputs
   the forbidden action    against a rubric

   different tests, different failure modes, constantly confused
```

Guardrails come in two kinds, and confusing them is how a programme passes audit
while missing harm.

**Operating guardrails** constrain *how the system runs*: all egress through the
gateway, privileged tools gated below L3, every action logged. They are testable
today, cheap to verify, and produce clean evidence.

**Outcome guardrails** constrain *what results are acceptable*: no unrecoverable
customer data loss, no increase in customer-facing incidents, no disparate
outcomes across segments. They matter more and most need a measurement you do
not yet have.

The failure is not choosing one. It is shipping only the first column, reporting
it as coverage, and never labelling the second column as unmeasured.

## 3 · The procedure, as a skill

Four operating guardrails are enforceable today; three outcome guardrails are enforceable only where a measurement exists. The skill classifies each rule, specifies the missing measurements, and counts coverage twice — against what shipped and against what was agreed.

### The skill — [`skills/grc/guardrail-specification/SKILL.md`](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/grc/guardrail-specification/SKILL.md)

```yaml
name: guardrail-specification
description: >-
  Separate operating guardrails, which are enforceable today, from outcome
  guardrails, which need a measurement before they mean anything — and specify
  that measurement. Use when a policy contains a rule nobody can enforce.
allowed-tools: Read, Grep, Glob
```

# An outcome guardrail with no measurement is a wish

Operating guardrails constrain what the system may do: which tools, which data,
which actions need approval. They are enforceable today. Outcome guardrails
constrain what the system may cause — no discriminatory decisions, no
misleading advice — and they are enforceable only where somebody has specified
the measurement.

## When to use this

Writing an AI policy, reviewing one, or explaining why a coverage figure of 100%
is counting only the rules that shipped.

## Procedure

**1 — Classify every rule.** Does it constrain the system's behaviour, or the
outcome of that behaviour? The test is whether it can be checked at the moment
of action.

**2 — For each operating guardrail, name the enforcement point.** The gateway,
the tool policy, the approval gate. If there is not one, it is aspirational and
should be reported that way.

**3 — For each outcome guardrail, specify the measurement.** The metric, the
population, the threshold, the cadence, and who reviews it. Four of those five
being present is still not a guardrail.

**4 — Count coverage both ways.** Against the rules that shipped, and against
all the rules that were agreed. The first is usually 100% and the second is
usually about half, and the gap is the honest programme statement.

**5 — Report the unmeasurable ones as open commitments** with an owner and a
date. Leaving them in the policy unmarked is how a policy stops being read.

## Output contract

```json
{
  "rules": [{"text": "str", "kind": "operating|outcome",
             "enforcement_point": "str|null",
             "measurement": {"metric": "str", "population": "str", "threshold": 0.0,
                             "cadence": "str", "reviewer": "str"}}],
  "coverage": {"of_shipped": 1.0, "of_agreed": 0.0},
  "open_commitments": [{"rule": "str", "owner": "str", "due": "str"}]
}
```

## Failure modes

- **Counting only what shipped.** The number is 100% by construction.
- **An outcome guardrail with a metric and no threshold.** Nothing fails.
- **Leaving unmeasurable rules unmarked.** The policy loses credibility as a
  whole.

In [ ]:
# The code is not in this notebook. It is this file in the repository:
#   https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/grc/guardrail-specification/scripts/guardrail_specification.py
SCRIPT = "skills/grc/guardrail-specification/scripts/guardrail_specification.py"
REPO = "https://github.com/spbreed/cyber-commons"
BRANCH = "claude/vulnbench-setup-scheduling-81aqov"

import glob, os, subprocess, sys

CLONE = "/kaggle/working/cyber-commons"
_root = next((r for r in (".", "..", "../..", CLONE)
              if os.path.isfile(os.path.join(r, SCRIPT))), None)

if _root is None:
    # --filter=blob:none --sparse fetches the tree without the history or the
    # notebooks; `sparse-checkout set skills` then materialises only what runs.
    _c = subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none",
                         "--sparse", "--branch", BRANCH, REPO, CLONE],
                        capture_output=True, text=True)
    if _c.returncode:
        raise SystemExit(
            "could not fetch the skills: " + _c.stderr.strip()[-300:] +
            "\nOn Kaggle this needs Internet on in the notebook settings, which "
            "needs a phone-verified account. Without one, attach the dataset "
            "cybercommons/cyber-commons-skills instead — it holds the same tree.")
    subprocess.run(["git", "-C", CLONE, "sparse-checkout", "set", "skills"],
                   capture_output=True, text=True)
    _root = CLONE

_out = subprocess.run([sys.executable, os.path.join(_root, SCRIPT)],
                      capture_output=True, text=True,
                      env=dict(os.environ,
                               PYTHONPATH=os.path.join(_root, "skills/_runtime"),
                               PYTHONHASHSEED="0"))
print(_out.stdout, end="")
if _out.returncode:
    raise SystemExit(_out.stderr.strip()[-2000:])

## What you just proved

Four operating guardrails are all enforceable today; three outcome guardrails are enforceable only where a measurement exists. Counting only what shipped gives 100% coverage; counting all agreed guardrails gives 71%. One outcome guardrail is fully specified and enforceable; the other is labelled an aspiration and excluded from coverage.

## Your turn

Pick one outcome guardrail your programme has agreed and specify its metric, threshold, source and cadence precisely enough that someone could dispute the result. If you cannot, say so in the coverage report rather than counting it.

---

**Next → [E1.7 · Continuous control verification](https://spbreed.github.io/cyber-commons/lessons/E1.7.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/E1.6.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/E1.6.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*